# Modeling Strategy Implementation

### Covered Workflow
- Load training and test data
- Load Stage 3 feature-set scores from `outputs/stage3_feature_set_scores.csv`
- Compare model families across the precomputed feature subsets
- Perform per-model hyperparameter optimization (HPO) for all candidate model families and persist results to `outputs/hpo/`
- Tune the final business score using out-of-fold probabilities
- Fit the final model and export ranked predictions

### Inputs and Outputs
- Input ranking artifact: `outputs/stage3_feature_set_scores.csv`
- HPO artifacts: `outputs/hpo/{model_name}_hpo.json`, `outputs/hpo/hpo_summary.json`
- Optimal threshold (from baseline): `outputs/optimal_threshold.json`
- Output predictions: `outputs/model_predictions.csv`

## Key Parameters

| Parameter | Default | Description |
|---|---|---|
| candidate_feature_sizes | (1, 3, 5, 8, 10, 15, 20, all) | Top-k feature subsets evaluated in modeling |
| modeling_cv_folds | 5 | Cross-validation folds used for model comparison |
| max_test_targets | 1000 | Maximum ranked test samples to retain |
| random_state | 42 | Random seed |
| include_xgboost | False | Optional XGBoost factory when the dependency is available |

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from tqdm.notebook import tqdm

from cost_effective.dataset import custom_scorer, get_classifier
from cost_effective.models import (
    build_model_factories,
    build_profit_curve,
    compare_models_on_feature_sets,
    compute_oof_probabilities,
    fit_final_model_and_predict,
)

PROJECT_ROOT: Path = Path().resolve().parent
DATA_PATH: Path = PROJECT_ROOT / "data"
OUTPUTS_PATH: Path = PROJECT_ROOT / "outputs"


LOGISTIC_PARAM_GRID: dict[str, list] = {
    "logisticregression__C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    "logisticregression__penalty": ["l1", "l2"],
}

LGB_PARAM_DIST: dict[str, list] = {
    "num_leaves": [15, 31, 63, 127],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "n_estimators": [100, 200, 400],
    "max_depth": [3, 4, 6, 8, -1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
}

XGB_PARAM_DIST: dict[str, list] = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 6, 8],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
}

In [3]:
try:
    with (OUTPUTS_PATH / "optimal_threshold.json").open("r") as f:
        threshold_config = json.load(f)
    optimal_threshold = threshold_config["optimal_threshold"]
    print(f"✓ Optimal threshold loaded from baseline: {optimal_threshold:.3f}")
except FileNotFoundError:
    print("⚠ Optimal threshold file not found; defaulting to 0.5")
    optimal_threshold = 0.5

✓ Optimal threshold loaded from baseline: 0.290


In [4]:
X_train = pd.read_csv(
    DATA_PATH / "x_train.txt",
    sep=r"\s+",
    header=None,
    skiprows=1,
    low_memory=False,
).apply(pd.to_numeric, errors="coerce")
X_train.fillna(0.0)
X_train.columns = [f"var_{i}" for i in range(X_train.shape[1])]

y_train = pd.read_csv(DATA_PATH / "y_train.txt", header=None, skiprows=1).iloc[:, 0].astype(int)

X_test = pd.read_csv(
    DATA_PATH / "x_test.txt",
    sep=r"\s+",
    header=None,
    skiprows=1,
    low_memory=False,
).apply(pd.to_numeric, errors="coerce")
X_test.fillna(0.0)
X_test.columns = [f"var_{i}" for i in range(X_test.shape[1])]

stage3_scores = pd.read_csv(OUTPUTS_PATH / "stage3_feature_set_scores.csv")
stage3_scores["features"] = stage3_scores["features"].apply(ast.literal_eval)

stage2_row = stage3_scores.loc[stage3_scores["feature_count"].idxmax()]
stage2_features = stage2_row["features"]
feature_set_candidates = {
    row.feature_set_name: row.features for row in stage3_scores.itertuples(index=False)
}
X_stage2 = X_train[stage2_features]

print(f"✓ Loaded training data: {X_train.shape}")
print(f"✓ Loaded test data: {X_test.shape}")
print(f"✓ Loaded labels: {len(y_train)} rows")
print(f"✓ Loaded Stage 3 score rows: {len(stage3_scores)}")
print(f"✓ Loaded Stage 2 feature set: {len(stage2_features)} features")

✓ Loaded training data: (5000, 500)
✓ Loaded test data: (5000, 500)
✓ Loaded labels: 5000 rows
✓ Loaded Stage 3 score rows: 8
✓ Loaded Stage 2 feature set: 26 features


In [5]:
base_factories = build_model_factories(y_train)

kw = {
    "scoring": "roc_auc",
    "verbose": 2,
    "n_jobs": -1,
    "cv": 5,
    "refit": "roc_auc",
}
n_iter = 50

hpo_results = {}
tuned_factories = {}
hpo_out_dir = OUTPUTS_PATH / "hpo"
hpo_out_dir.mkdir(parents=True, exist_ok=True)

pbar = tqdm(base_factories.items(), desc="Hyperparameter Optimization", unit="model")
for model_name, factory in pbar:
    pbar.set_postfix_str(f"Tuning {model_name}")

    if model_name == "logistic_regression":
        base_est = factory()
        # Grid over pipeline parameters
        gs = GridSearchCV(base_est, LOGISTIC_PARAM_GRID, **kw)
        gs.fit(X_stage2, y_train)
        best = gs.best_estimator_
        best_params = gs.best_params_

    elif model_name == "lightgbm":
        base_est = get_classifier(y_train.to_numpy())
        rs = RandomizedSearchCV(base_est, LGB_PARAM_DIST, n_iter=n_iter, random_state=42, **kw)
        rs.fit(X_stage2, y_train)
        best = rs.best_estimator_
        best_params = rs.best_params_

    elif model_name == "xgboost":
        base_est = factory()
        rs = RandomizedSearchCV(base_est, XGB_PARAM_DIST, n_iter=n_iter, random_state=42, **kw)
        rs.fit(X_stage2, y_train)
        best = rs.best_estimator_
        best_params = rs.best_params_

    else:
        raise ValueError(f"HPO not configured for model: {model_name}")

    # Save HPO results and create a cloning factory for the best estimator
    hpo_results[model_name] = {
        "best_params": best_params,
        "score": float(custom_scorer(best, X_stage2.to_numpy(), y_train.to_numpy())),
    }
    # Persist per-model HPO summary
    with (hpo_out_dir / f"{model_name}_hpo.json").open("w") as fh:
        json.dump(hpo_results[model_name], fh, indent=2)

    # Factory that clones the fitted estimator
    tuned_factories[model_name] = (lambda est=best: lambda *_: clone(est))()

# Save aggregate HPO results
with (hpo_out_dir / "hpo_summary.json").open("w") as fh:
    json.dump(hpo_results, fh, indent=2)

# Run model comparison using tuned factories
model_comparison = compare_models_on_feature_sets(
    X_stage2,
    y_train,
    feature_set_candidates,
    estimator_factories=tuned_factories,
    cv=5,
    threshold=optimal_threshold,
)


# Attach best hyperparameters to the comparison table where available
def _params_for(model_name):
    return hpo_results.get(model_name, {}).get("best_params", {})


model_comparison["best_hyperparams"] = model_comparison["model_name"].apply(_params_for)

best_model_row = model_comparison.iloc[0] if len(model_comparison) else None
best_model_name = str(best_model_row["model_name"]) if best_model_row is not None else None
best_feature_set_name = (
    str(best_model_row["feature_set_name"]) if best_model_row is not None else None
)
best_features = (
    feature_set_candidates[best_feature_set_name] if best_feature_set_name is not None else []
)

print(f"Best model: {best_model_name}")
print(f"Best feature set: {best_feature_set_name} ({len(best_features)} features)")
model_comparison

Hyperparameter Optimization:   0%|          | 0/3 [00:00<?, ?model/s]

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END colsample_bytree=0.6, learning_rate=0.1, max_depth=8, n_estimators=200, num_leaves=63, subsample=1.0; total time=   6.2s
[CV] END colsample_bytree=0.6, learning_rate=0.1, max_depth=8, n_estimators=200, num_leaves=63, subsample=1.0; total time=   6.4s
[CV] END colsample_bytree=0.6, learning_rate=0.1, max_depth=8, n_estimators=200, num_leaves=63, subsample=1.0; total time=   6.7s
[CV] END colsample_bytree=0.6, learning_rate=0.1, max_depth=8, n_estimators=200, num_leaves=63, subsample=1.0; total time=   6.7s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=6, n_estimators=400, num_leaves=31, subsample=0.6; total time=   6.9s
[CV] END colsample_bytree=0.6, learning_rate=0.1, max_depth=8, n_estimators=200, num_leaves=63, subsample=1.0; total time=   6.6s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=6, n_estimators=400, num_leaves=31, subsample=0.6; total time=   7.2s
[CV] END colsample_bytree=1.

,model_name,feature_set_name,feature_count,cv_score_mean,cv_score_std,f1_score,business_score_no_var_penalty,best_hyperparams
0,lightgbm,top_01,1,2264.0,7.348469,0.664530,2464.0,"{'subsample': 0.6, 'num_leaves': 15, 'n_estima..."
1,logistic_regression,top_01,1,2264.0,7.348469,0.664530,2464.0,"{'logisticregression__C': 0.01, 'logisticregre..."
2,xgboost,top_01,1,2264.0,7.348469,0.664530,2464.0,"{'subsample': 0.8, 'n_estimators': 400, 'max_d..."
3,lightgbm,top_03,3,1874.0,15.620499,0.665419,2474.0,"{'subsample': 0.6, 'num_leaves': 15, 'n_estima..."
4,logistic_regression,top_03,3,1864.0,7.348469,0.664530,2464.0,"{'logisticregression__C': 0.01, 'logisticregre..."
5,xgboost,top_03,3,1864.0,7.348469,0.664530,2464.0,"{'subsample': 0.8, 'n_estimators': 400, 'max_d..."
6,lightgbm,top_05,5,1483.0,27.129320,0.666212,2483.0,"{'subsample': 0.6, 'num_leaves': 15, 'n_estima..."
7,xgboost,top_05,5,1479.0,21.540659,0.665854,2479.0,"{'subsample': 0.8, 'n_estimators': 400, 'max_d..."
8,logistic_regression,top_05,5,1464.0,22.449944,0.664520,2464.0,"{'logisticregression__C': 0.01, 'logisticregre..."
9,lightgbm,top_08,8,876.0,29.051678,0.665585,2476.0,"{'subsample': 0.6, 'num_leaves': 15, 'n_estima..."


In [6]:
oof_probabilities = compute_oof_probabilities(
    X_stage2[best_features],
    y_train,
    estimator_factory=tuned_factories.get(best_model_name, base_factories.get(best_model_name)),
    cv=5,
)
profit_curve = build_profit_curve(
    y_train,
    oof_probabilities,
    feature_count=len(best_features),
    max_targets=1000,
)

print(
    f"Best OOF cutoff: k={profit_curve.best_k}, threshold={profit_curve.best_threshold:.4f}, "
    f"score={profit_curve.best_score:.2f}"
)
profit_curve.curve.head(10)

Best OOF cutoff: k=997, threshold=0.5438, score=2600.00


,k,tp,fp,score,threshold
0,1,1,0,-190,0.6595
1,2,1,1,-195,0.6595
2,3,2,1,-185,0.6595
3,4,3,1,-175,0.6595
4,5,4,1,-165,0.6595
5,6,4,2,-170,0.6595
6,7,5,2,-160,0.6595
7,8,6,2,-150,0.6595
8,9,6,3,-155,0.6595
9,10,6,4,-160,0.6595


In [7]:
final_prediction_result = fit_final_model_and_predict(
    X_train=X_stage2,
    y_train=y_train,
    X_test=X_test,
    selected_features=best_features,
    estimator_factory=tuned_factories.get(best_model_name, base_factories.get(best_model_name)),
    max_targets=1000,
)

# Filter predictions by threshold and rank by probability
predicted_probs = final_prediction_result.probabilities
threshold_mask = predicted_probs > profit_curve.best_threshold
above_threshold_indices = np.where(threshold_mask)[0]

# Sort by probability (descending)
sorted_indices = above_threshold_indices[np.argsort(-predicted_probs[above_threshold_indices])]
sorted_indices = sorted_indices[:1000]  # Limit to top 1000 if more are above threshold

final_prediction_frame = pd.DataFrame({
    "rank": np.arange(1, len(sorted_indices) + 1),
    "sample_index": sorted_indices,
    "probability": predicted_probs[sorted_indices],
})

final_prediction_frame.to_csv(OUTPUTS_PATH / "model_predictions.csv", index=False)

print(f"Samples above threshold: {len(sorted_indices)}")
final_prediction_frame.head()

Samples above threshold: 852


,rank,sample_index,probability
0,1,3207,0.692458
1,2,4093,0.692458
2,3,2034,0.692458
3,4,2743,0.692458
4,5,4250,0.692458
